<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/14_scaling_with_pairwise_comparisons_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Scaling Texts with Pairwise Comparisons

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Many concepts in the social sciences are a matter of degree: how uncivil is a comment, how far left
is a manifesto, how strong is an argument. A classifier puts texts into boxes; here we want a
**scale**. This notebook has an LLM place online comments on a toxicity scale in two ways: by rating
each comment on its own, and by comparing comments two at a time and turning those comparisons into
a scale with the **Bradley–Terry model**. Both scales are checked against the share of human raters
who found each comment toxic.

> ⚠️ **Content note:** the comments come from real comment sections and include insults and
> profanity. We leave out comments that raters flagged as identity attacks, threats or sexually
> explicit, but the rest is unfiltered.

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

In this notebook you will:

- see why comparing two texts is often easier than rating one, for people and for models
- rate comments directly, reading a score from the model's probabilities
- compare comments in pairs, in both orders, and measure the model's position bias
- turn comparisons into a scale with the Bradley–Terry model
- validate both scales against human raters, check how many comparisons you need, and place new comments on an existing scale

**Contents**

1. Why compare?
2. The data
3. The model
4. Direct ratings
5. Pairwise comparisons
6. From comparisons to a scale: Bradley–Terry
7. Validation
8. How many comparisons, and how reliable?
9. Placing new texts on the scale
10. Wrap-up

There are 5 exercises along the way, marked ✏️. The solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q transformers accelerate

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
set_seed(SEED)
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 150)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → T4 GPU")

---
## 1. Why compare?

Ask ten people to rate a comment from 0 to 9 and you get ten different scales: one person's 6 is
another's 8. Ask them which of two comments is ruder and they mostly agree. That's the idea behind
**pairwise comparison**, used for human coding of political texts (Carlson & Montgomery, 2017) and,
more recently, with LLMs (Wu et al., 2023). The comparisons are then combined into one scale with a
model from the 1950s, **Bradley–Terry**: the same idea as rating chess players from who beat whom.

| | Direct rating | Pairwise comparison |
|---|---|---|
| Prompts | one per text | several per text, in pairs |
| What the model judges | an absolute number | which of two is more X |
| Weak spot | every rater (and model) uses the scale differently | needs many prompts, and a model to combine them |
| Typical use | quick, large corpora | careful measurement of a sample |

We'll do both on the same comments and see which agrees better with human raters.

---
## 2. The data

**Civil Comments** (Borkan et al., 2019): about two million comments from the comment sections of
news sites, posted 2015–2017. Each comment was shown to several crowd raters. Its `toxicity` score is
the **share of raters** who found it toxic, so 0.3 means three in ten. We use the validation split.

In [ ]:
path = hf_hub_download(repo_id="google/civil_comments", repo_type="dataset",
                       revision="refs/convert/parquet", filename="default/validation/0000.parquet")
cc = pd.read_parquet(path)
print(f"{len(cc):,} comments")
cc["toxicity"].describe().round(3)

Most comments aren't toxic at all, so a random sample would be almost all zeros. We keep comments of
15 to 80 words, leave out those that raters flagged as identity attacks, threats or sexually explicit
(see the content note), and draw **10 comments from each of six toxicity bands**.

In [ ]:
words = cc["text"].str.split().str.len()
keep = cc[words.between(15, 80) & (cc[["identity_attack", "threat", "sexual_explicit"]].max(axis=1) < 0.2)].copy()
keep["text"] = keep["text"].str.replace(r"\s+", " ", regex=True).str.strip()
keep["band"] = pd.cut(keep["toxicity"], [0, 0.05, 0.2, 0.4, 0.6, 0.8, 1.01], right=False)

sample = (keep.groupby("band", observed=True).sample(10, random_state=SEED)
              .sample(frac=1, random_state=SEED).reset_index(names="row"))
N = len(sample)
print(N, "comments")
sample.groupby("band", observed=True)["toxicity"].agg(["count", "mean"]).round(2)

In [ ]:
for _, row in sample.sort_values("toxicity").iloc[[0, N // 2, -1]].iterrows():
    print(f"toxicity {row.toxicity:.2f}: {row.text[:220]}\n")

---
## 3. The model

Qwen2.5 1.5B Instruct, as in the Day 4 notebook. We never let it write. As with label scoring on
Day 4, we read the probability of each possible answer token directly: the digits 0 to 9 for a rating,
and A or B for a comparison.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="auto")

DIGITS = list("0123456789")
for options in [DIGITS, ["A", "B"]]:                       # each answer must start with its own token
    ids = [tokenizer.encode(o, add_special_tokens=False)[0] for o in options]
    assert len(set(ids)) == len(options), options


def answer_probs(messages, options):
    """Probability of each option as the model's first answer token, renormalized over the options."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    ids = [tokenizer.encode(o, add_special_tokens=False)[0] for o in options]
    with torch.no_grad():
        logits = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).logits[0, -1]
    return torch.softmax(logits[ids].float(), dim=-1).cpu().numpy()


DEFINITION = ("A toxic comment is rude, disrespectful or unreasonable: the kind of comment that makes people "
              "want to leave a discussion.")

---
## 4. Direct ratings

One prompt per comment: rate it from 0 to 9. Instead of taking the single most likely digit, we take
the **expected rating** over all ten digits. A comment the model can't decide on, split between 2 and
7, then lands in between instead of jumping to one end.

In [ ]:
def rate(text):
    messages = [{"role": "system", "content": DEFINITION + " Rate the comment from 0 (not toxic at all) "
                 "to 9 (extremely toxic). Reply with one digit."},
                {"role": "user", "content": f"Comment: {text}"}]
    return float(answer_probs(messages, DIGITS) @ np.arange(10))


start = time.time()
sample["rating"] = [rate(t) for t in sample["text"]]
print(f"{N} ratings in {time.time() - start:.0f} s")
sample[["text", "toxicity", "rating"]].head()

---
## 5. Pairwise comparisons

Each **round** shuffles the comments and pairs them up, so every comment meets one other comment per
round. Eight rounds give every comment eight opponents: 240 pairs.

We ask every pair **twice**, once in each order. LLMs, like people, tend to favor the first or the
second option regardless of content; asking both ways lets us measure that and cancel it out.

In [ ]:
ROUNDS = 8
rows = []
for r in range(ROUNDS):
    order = rng.permutation(N)
    rows += [(order[k], order[k + 1], r) for k in range(0, N - 1, 2)]
pairs = pd.DataFrame(rows, columns=["i", "j", "round"])
print(len(pairs), "pairs,", 2 * len(pairs), "prompts")


def compare(text_a, text_b):
    """Probability that the model calls comment A the more toxic one."""
    messages = [{"role": "system", "content": DEFINITION + " You will see two comments, A and B. "
                 "Which one is more toxic? Reply with A or B."},
                {"role": "user", "content": f"Comment A: {text_a}\n\nComment B: {text_b}"}]
    return answer_probs(messages, ["A", "B"])[0]


texts = sample["text"].tolist()
start = time.time()
pairs["p_ij"] = [compare(texts[i], texts[j]) for i, j in zip(pairs.i, pairs.j)]   # i shown first
pairs["p_ji"] = [compare(texts[j], texts[i]) for i, j in zip(pairs.i, pairs.j)]   # j shown first
print(f"{2 * len(pairs)} comparisons in {time.time() - start:.0f} s")

If comment *i* really is more toxic, `p_ij` (i shown first) should be high and `p_ji` (j shown first)
low. If both are high, the model simply picked whichever came first.

In [ ]:
first = np.concatenate([pairs.p_ij, pairs.p_ji]).mean()
order_decided = ((pairs.p_ij > 0.5) == (pairs.p_ji > 0.5)).mean()
print(f"Average probability for the comment shown first: {first:.2f}   (no position bias: 0.50)")
print(f"Pairs where the order, not the content, decided the verdict: {order_decided:.0%}")

pairs["p"] = (pairs.p_ij + (1 - pairs.p_ji)) / 2         # P(i more toxic than j), both orders averaged
pairs.head()

---
## 6. From comparisons to a scale: Bradley–Terry

Bradley–Terry gives every comment a position $\theta$ on the scale, and assumes

$$P(i \text{ is judged more toxic than } j) = \frac{e^{\theta_i}}{e^{\theta_i} + e^{\theta_j}}.$$

A comment that "wins" against comments that themselves win a lot ends up high. The fit takes a few
lines with Hunter's (2004) algorithm. Two details:

- our outcomes are **probabilities** (0.8 means "probably i"), which Bradley–Terry can use as
  fractional wins, so uncertain comparisons count less;
- a small **prior**, half a win and half a loss against an average comment, keeps a comment that won
  (or lost) every comparison from running off to infinity.

In [ ]:
def bradley_terry(i, j, p, n, prior=0.5, iters=1000):
    """Positions on a log scale (centered at 0) from pairwise outcomes; p = probability that i beat j."""
    i, j, p = np.asarray(i), np.asarray(j), np.asarray(p, dtype=float)
    wins = np.full(n, prior)
    np.add.at(wins, i, p)
    np.add.at(wins, j, 1 - p)
    strength = np.ones(n)
    for _ in range(iters):
        denom = np.full(n, 2 * prior / (strength + 1))     # the prior's games against an average comment
        pair_term = 1 / (strength[i] + strength[j])
        np.add.at(denom, i, pair_term)
        np.add.at(denom, j, pair_term)
        strength = wins / denom
        strength /= np.exp(np.log(strength).mean())
    return np.log(strength)


sample["bt"] = bradley_terry(pairs.i, pairs.j, pairs.p, N)
sample.sort_values("bt")[["text", "toxicity", "bt"]].iloc[[0, 1, 2, -3, -2, -1]]

---
## 7. Validation

Both scales against the crowd. We use **Spearman's** rank correlation as the main measure: the scales
have different units, and what matters is whether they put the comments in the same order.

In [ ]:
def report(name, scores):
    rho = spearmanr(scores, sample["toxicity"])[0]
    r = pearsonr(scores, sample["toxicity"])[0]
    print(f"{name:<24} Spearman {rho:.2f}   Pearson {r:.2f}")


report("Direct rating", sample["rating"])
report("Pairwise + Bradley-Terry", sample["bt"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label in [(axes[0], "rating", "Expected rating (0-9)"), (axes[1], "bt", "Bradley–Terry position")]:
    ax.scatter(sample["toxicity"], sample[col], color="#34B233", alpha=0.7)
    ax.set_xlabel("Share of human raters who found it toxic")
    ax.set_ylabel(label)
plt.tight_layout()
plt.show()

Another way to look at it: on pairs where the human raters clearly disagree (their scores differ by
at least 0.3), how often does the model pick the same comment as more toxic?

In [ ]:
crowd = sample["toxicity"].to_numpy()
diff = crowd[pairs.i] - crowd[pairs.j]
clear = np.abs(diff) >= 0.3
agree = ((pairs.p > 0.5) == (diff > 0))[clear].mean()
print(f"{clear.sum()} pairs with a clear difference: the model agrees with the raters on {agree:.0%}")

Keep in mind that the crowd score is a measurement too. A comment seen by only a few raters gets a
noisy score, and "toxic" meant different things to different raters. The disagreements are where you
learn the most.

> **✏️ Exercise 1**
>
> Find the six comments where the pairwise scale and the crowd disagree most. Compare **ranks**, not
> raw scores (`.rank(pct=True)` gives each comment's percentile). Read them: is the model wrong, are
> the raters, or is it a matter of definition?

In [ ]:
# Your code here

---
## 8. How many comparisons, and how reliable?

Pairwise comparison is expensive: 60 comments × 8 opponents × 2 orders is already 480 prompts, and
the number grows with every comment you add. So: how much do the second order and the later rounds
actually buy you?

> **✏️ Exercise 2**
>
> Fit the scale using **only one order** (`pairs.p_ij`, the comment shown first). How does it compare
> with the scale from both orders? Relate your answer to the position bias you measured in Section 5.

In [ ]:
# Your code here

> **✏️ Exercise 3**
>
> Refit the scale with the first 1, 2, ..., 8 rounds (`pairs[pairs["round"] < r]`) and plot the
> Spearman correlation with the crowd against the number of rounds. Where does it level off? Draw the
> direct rating's correlation as a horizontal line for comparison.

In [ ]:
# Your code here

> **✏️ Exercise 4**
>
> **Split-half reliability.** Fit one scale on the even rounds and one on the odd rounds. How strongly
> do the two scales agree with each other? Why does this number matter for the correlation with the
> crowd?

In [ ]:
# Your code here

---
## 9. Placing new texts on the scale

Comparing every new comment with every comment you already have doesn't scale. Instead, pick a set of
**anchors** spread along the scale, compare a new comment with each anchor only, and find the position
that best explains those outcomes, keeping the anchors where they are.

> **✏️ Exercise 5**
>
> Take ten anchors spread evenly over the Bradley–Terry scale, and one new comment from each toxicity
> band (comments not in `sample`; use the `row` column to exclude them). Compare each new comment
> with every anchor in both orders, and find its position: try positions from −6 to 6 and keep the one
> with the highest Bradley–Terry likelihood. Does the order of the new comments match their crowd
> scores?

In [ ]:
# Your code here

---
## 10. Wrap-up

- **Degrees, not boxes.** When your concept is a matter of degree, a scale throws away less than a
  category, and pairwise comparison is a robust way to build one.
- **Probabilities beat sampled answers.** Reading the model's probabilities gives smooth ratings
  and soft wins, and makes the result the same every time you run it.
- **Ask both orders.** Measure the position bias and report it.
- **Validate** against human judgments, ideally including human pairwise comparisons on a subset
  (Bonus 5 has the tools for agreement).
- **Report** the model, the full prompt with your definition, the pairing design, the number of
  comparisons, and how you fit the scale.
- **Scale up** with anchors, or by distilling the scale into a small model.

### Further reading

- Bradley, R. A., & Terry, M. E. (1952). Rank analysis of incomplete block designs: I. The method of
  paired comparisons. *Biometrika, 39*(3/4), 324–345.
- Hunter, D. R. (2004). MM algorithms for generalized Bradley–Terry models. *The Annals of
  Statistics, 32*(1), 384–406.
- Carlson, D., & Montgomery, J. M. (2017). A pairwise comparison framework for fast, flexible, and
  reliable human coding of political texts. *American Political Science Review, 111*(4), 835–843.
- Wu, P. Y., Nagler, J., Tucker, J. A., & Messing, S. (2023). Large language models can be used to
  estimate the latent positions of politicians. *arXiv preprint*.
- Benoit, K., De Marchi, S., Laver, C., Laver, M., & Ma, J. (2026). Using large language models to
  analyze political texts through natural language understanding. *American Journal of Political
  Science*.
- Borkan, D., Dixon, L., Sorensen, J., Thain, N., & Vasserman, L. (2019). Nuanced metrics for
  measuring unintended bias with real data for text classification. *Companion Proceedings of the
  2019 World Wide Web Conference*.
- Zheng, L., et al. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena. *NeurIPS*.